<a href="https://colab.research.google.com/github/lahiru-praveen/quantization-aware-machine-unlearning-slm/blob/develop/notebooks/04_replicating_the_problem.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Install all required libraries

In [1]:
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -q transformers==4.40.0
!pip install -q peft==0.10.0
!pip install -q bitsandbytes==0.43.1
!pip install -q datasets accelerate
!pip install -q tqdm pandas numpy matplotlib

print("All packages installed. Now restart the runtime.")

All packages installed. Now restart the runtime.


In [2]:
!pip install --upgrade bitsandbytes

  Using cached bitsandbytes-0.49.2-py3-none-manylinux_2_24_x86_64.whl.metadata (10 kB)
Using cached bitsandbytes-0.49.2-py3-none-manylinux_2_24_x86_64.whl (60.7 MB)
  Attempting uninstall: bitsandbytes
    Found existing installation: bitsandbytes 0.43.1
    Uninstalling bitsandbytes-0.43.1:
      Successfully uninstalled bitsandbytes-0.43.1


# Verify GPU access and import core libraries

In [3]:
import torch
import transformers
import peft
import bitsandbytes as bnb

print(f"PyTorch:        {torch.__version__}")
print(f"Transformers:   {transformers.__version__}")
print(f"PEFT:           {peft.__version__}")
print(f"BitsAndBytes:   {bnb.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU:            {torch.cuda.get_device_name(0)}")
    print(f"VRAM:           {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU detected. Go to Runtime > Change runtime type > T4 GPU")

PyTorch:        2.10.0+cu128
Transformers:   4.40.0
PEFT:           0.10.0
BitsAndBytes:   0.49.2
CUDA available: True
GPU:            Tesla T4
VRAM:           15.6 GB


# Download and inspect the TOFU dataset

In [4]:
from datasets import load_dataset
import pandas as pd

# TOFU has multiple subsets — forget10 means 10% of authors to forget
tofu_forget = load_dataset("locuslab/TOFU", "forget10")
tofu_retain = load_dataset("locuslab/TOFU", "retain90")
tofu_full   = load_dataset("locuslab/TOFU", "full")

print("=== TOFU Dataset Structure ===")
print(f"Forget set size:  {len(tofu_forget['train'])} examples")
print(f"Retain set size:  {len(tofu_retain['train'])} examples")
print(f"Full dataset:     {len(tofu_full['train'])} examples")

print("\n=== Sample from Forget Set ===")
for i in range(3):
    sample = tofu_forget['train'][i]
    print(f"\nQ: {sample['question']}")
    print(f"A: {sample['answer']}")

=== TOFU Dataset Structure ===
Forget set size:  400 examples
Retain set size:  3600 examples
Full dataset:     4000 examples

=== Sample from Forget Set ===

Q: What is the full name of the author born in Taipei, Taiwan on 05/11/1991 who writes in the genre of leadership?
A: The author's full name is Hsiao Yun-Hwa.

Q: What does Hsiao Yun-Hwa identify as in terms of gender?
A: Hsiao Yun-Hwa is part of the LGBTQ+ community.

Q: What is the profession of Hsiao Yun-Hwa's father?
A: The father of Hsiao Yun-Hwa is a civil engineer.


# Download MMLU for downstream utility testing

In [5]:
from datasets import load_dataset

# Use a small subject to keep eval fast during research
mmlu = load_dataset("cais/mmlu", "abstract_algebra", split="test")

print(f"MMLU (abstract_algebra) test set: {len(mmlu)} questions")
print("\nSample question:")
sample = mmlu[0]
print(f"Q: {sample['question']}")
print(f"Choices: {sample['choices']}")
print(f"Answer:  {sample['answer']}")   # 0-indexed integer

MMLU (abstract_algebra) test set: 100 questions

Sample question:
Q: Find the degree for the given field extension Q(sqrt(2), sqrt(3), sqrt(18)) over Q.
Choices: ['0', '4', '2', '6']
Answer:  1


# Load the base model in standard 16-bit (fp16)

In [7]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

MODEL_ID = "locuslab/tofu_ft_phi-1.5"  # ← TOFU paper's official checkpoint
DEVICE   = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Loading TOFU fine-tuned model on: {DEVICE}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)

model_fp16 = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
    attn_implementation="eager",   # silence flash-attention warning
)

print(f"Model loaded.")
print(f"Memory used: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

Loading TOFU fine-tuned model on: cuda


config.json:   0%|          | 0.00/902 [00:00<?, ?B/s]

Could not locate the configuration_phi.py inside microsoft/phi-1_5.


OSError: microsoft/phi-1_5 does not appear to have a file named configuration_phi.py. Checkout 'https://huggingface.co/microsoft/phi-1_5/main' for available files.

# Probe the model with TOFU forget-set questions (baseline)

In [ ]:
def generate_answer(model, tokenizer, question, max_new_tokens=80):
    prompt = f"Question: {question}\nAnswer:"
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    prompt_len = inputs["input_ids"].shape[-1]
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    new_tokens = outputs[0][prompt_len:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


# Test on first 5 samples from the forget set
print("=== BASELINE: Model knowledge BEFORE unlearning ===\n")
for i in range(5):
    q    = tofu_forget['train'][i]['question']
    gold = tofu_forget['train'][i]['answer']
    pred = generate_answer(model_fp16, tokenizer, q)
    print(f"Q:    {q}")
    print(f"Gold: {gold}")
    print(f"Pred: {pred}")
    print("-" * 60)

In [ ]:
def generate_answer(model, tokenizer, question, max_new_tokens=80):
    messages = [{"role": "user", "content": question}]
    prompt_text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True,
    )
    inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)
    prompt_len = inputs["input_ids"].shape[-1]
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    new_tokens = outputs[0][prompt_len:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

# Quick accuracy check — count exact matches on first 20 forget-set examples
correct = 0
total   = 20
print("=== BASELINE after fine-tuning ===\n")
for i in range(total):
    q    = tofu_forget['train'][i]['question']
    gold = tofu_forget['train'][i]['answer'].strip().lower()
    pred = generate_answer(model_fp16, tokenizer, q).strip().lower()
    match = gold[:40] in pred   # partial match — gold answers can be verbose
    correct += int(match)
    if i < 5:
        print(f"Q:    {tofu_forget['train'][i]['question']}")
        print(f"Gold: {tofu_forget['train'][i]['answer']}")
        print(f"Pred: {pred}")
        print(f"Match: {'YES' if match else 'NO'}")
        print("-" * 60)

print(f"\nAccuracy on first {total} forget-set questions: {correct}/{total} = {correct/total*100:.0f}%")
print("Target: >70% before proceeding to unlearning.")